# 6. Siapa yang untung kalau harga bahan mie ayam naik? (Bagian 1: siapkan data)

Di notebook 2-5 aku hitung biaya bahan semangkuk mie ayam. Sekarang pertanyaannya dibalik: **kalau biaya bahan naik, perusahaan mana yang kelihatannya ikut untung?**

Aku pakai harga saham 5 perusahaan pangan di bursa (IDX) sebagai "petunjuk":

| Ticker | Perusahaan | Kenapa dipilih |
|---|---|---|
| CPIN | Charoen Pokphand | ayam (±70% biaya bahan) |
| JPFA | Japfa Comfeed | ayam |
| INDF | Indofood | mie instan (pembanding) |
| ICBP | Indofood CBP | mie instan (pembanding) |
| AALI | Astra Agro Lestari | sawit, bahan baku minyak goreng |

**Catatan jujur dari awal:** ini bukan analisis investasi. Harga saham dipengaruhi banyak hal selain harga pangan, dan datanya cuma ±69 bulan. Nanti hasilnya kubaca sebagai *korelasi*, bukan sebab-akibat.

Bagian 1 ini hanya menyiapkan data: biaya per porsi + harga saham, lalu digabung per bulan.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf   # di Kaggle: Settings > Internet harus ON

## Langkah 1: bangun ulang biaya per porsi (sama seperti notebook 5b)
Resep: ayam 60 g, bawang merah 10 g, bawang putih 5 g, minyak 20 ml. Terigu tidak dihitung (alasannya ada di 5b).

In [ ]:
DATA ='/kaggle/input/datasets/samuelgeraldo/bedah-mie-ayam-pakai-data-bukan-sumpit/harga_pangan_bersih.csv'
df = pd.read_csv(DATA)
wide = df.pivot(index='bulan', columns='komoditas', values='harga_rp')
wide.index = pd.to_datetime(wide.index)
wide = wide.interpolate(limit_direction='both')
wide = wide[wide.index < '2026-10-01']   # Okt 2026 belum lengkap

minyak = wide[['Minyak Goreng Kemasan Bermerk 1', 'Minyak Goreng Kemasan Bermerk 2']].mean(axis=1)
biaya = pd.DataFrame({
    'ayam': wide['Daging Ayam Ras Segar']*60/1000,
    'bawang_merah': wide['Bawang Merah Ukuran Sedang']*10/1000,
    'bawang_putih': wide['Bawang Putih Ukuran Sedang']*5/1000,
    'minyak': minyak*20/1000})
biaya['total'] = biaya.sum(axis=1)
print(biaya['total'].iloc[[0, -1]].round(0))   # harus 2763 (Jan 2021) dan 3613 (Sep 2026)
print(len(biaya), 'bulan')

## Langkah 2: ambil harga saham bulanan
`auto_adjust=True` artinya harga sudah disesuaikan dengan dividen dan stock split, jadi perubahan harganya tidak "palsu".

In [ ]:
tickers = ['CPIN.JK', 'JPFA.JK', 'INDF.JK', 'ICBP.JK', 'AALI.JK']
saham = yf.download(tickers, start='2021-01-01', end='2026-10-01',
                    interval='1mo', auto_adjust=True, progress=False)['Close']
saham.columns = [t.replace('.JK', '') for t in saham.columns]
saham.index = pd.to_datetime(saham.index).to_period('M').to_timestamp()   # samakan ke tanggal 1 tiap bulan
print(saham.shape, '| nilai kosong:', int(saham.isna().sum().sum()))
saham.head()

## Langkah 3: ubah semuanya jadi "perubahan % dari bulan sebelumnya"
Harga saham (ribuan rupiah) dan biaya mie (ribuan rupiah juga, tapi skalanya beda) tidak bisa dibandingkan langsung. Yang dibandingkan adalah **persen perubahannya**, sama seperti di notebook 5.

In [ ]:
gabung = pd.concat([biaya['total'].rename('biaya_mie'), saham], axis=1)
print('bulan di biaya:', len(biaya), '| bulan di saham:', len(saham), '| bulan gabungan:', len(gabung.dropna()))

perubahan = gabung.pct_change().dropna() * 100
perubahan.round(2).head(8)

## Langkah 4: lihat sekilas
Semua dinormalkan ke 100 di Jan 2021 supaya bisa ditaruh di satu grafik.

In [ ]:
indeks = gabung / gabung.iloc[0] * 100
fig, ax = plt.subplots(figsize=(11, 5))
indeks['biaya_mie'].plot(ax=ax, color='black', lw=3, label='biaya mie ayam')
indeks.drop(columns='biaya_mie').plot(ax=ax, lw=1.4)
ax.axhline(100, color='grey', ls=':')
ax.set_title('Biaya bahan mie ayam vs harga saham (Jan 2021 = 100)')
ax.set_ylabel('indeks'); ax.set_xlabel('')
ax.legend(ncol=3)
plt.tight_layout(); plt.show()

print((indeks.iloc[-1] - 100).round(1).sort_values())   # naik/turun berapa % sejak Jan 2021

In [ ]:
gabung.to_csv('biaya_vs_saham_bulanan.csv')   # disimpan untuk bagian 2

## Yang sudah dan belum
- Data biaya (69 bulan) dan harga saham (69 bulan) sudah digabung, tidak ada bulan kosong.
- Belum ada kesimpulan apa pun: grafik di atas cuma gambaran awal.

**Next (bagian 2):** korelasi perubahan bulanan biaya mie vs return tiap saham, dan apakah saham ayam bergerak lebih dulu / belakangan dari harga ayam. Caveat sampel kecil akan kutulis di sana.